In [4]:
import argparse
import datetime
import json
import numpy as np
import os
import time
from pathlib import Path

import torch
import torch.backends.cudnn as cudnn
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as transforms
import torchvision.datasets as datasets

import timm

assert timm.__version__ == "0.3.2"  # version check
import timm.optim.optim_factory as optim_factory

import util.misc as misc
from util.pos_embed import interpolate_pos_embed, interpolate_pos_embed_audio, interpolate_patch_embed_audio
from util.misc import NativeScalerWithGradNormCount as NativeScaler

from src import models_mae

from src.engine_pretrain import train_one_epoch, validate_one_epoch
from src.dataset import AudiosetDataset

# 
#from src.AEdataset import BirdsongDataset
#from src.AEdataset import generatePCNEMelSpec
import torch.nn as nn
from pathlib import Path

current_working_directory = Path.cwd()
script_dir = current_working_directory.joinpath('code')

In [3]:
args = {
    'batch_size': 128, 
    'epochs': 30, 
    'accum_iter': 1,
    'model': 'mae_vit_base_patch16',
    'weight_decay': 0.0001, 
    'lr': None, #
    'blr': 2e-4, # 
    'min_lr': 0.0, #
    'warmup_epochs': 3, 
    'norm_pix_loss': True, 
    'input_size': 128, 
    'low_freq':100, 
    'high_freq':11000, 
    'device': 'cuda',
    'seed': 0,
    'resume': '', 
    'start_epoch': 0,
    'num_workers': 3, # 10
    'pin_mem': True,
    'no_pin_mem': False,
    'save_every_epoch': 1, # 20
    'dataset': 'birdsong',
    'audio_exp': True, 

    'mean':-7.0850544,
    'std':2.853875,

    # augment
    'mask_ratio': 0.8, 
    'freqm': 0,
    'timem': 0,
    'mixup': 0,
    'mask_2d': False, 
    'mask_t_prob': 0.7, 
    'mask_f_prob': 0.3, 
    'noise': False,
    'roll_mag_aug': True, 
    ##

    'use_custom_patch': False, 

    'world_size': 1, # 1
    'local_rank': -1, # -1
    'dist_on_itp': False,
    'dist_url': 'env://',
    'rank': 0,
    'gpu': 0,
    'distributed': True,
    'dist_backend': 'nccl',
    ##

    'alpha': 0.0,
    'omega': 1.0,
    'mode': 0,
    'split_pos': False,
    'pos_trainable': False,
    'use_nce': False,
    'load_video': False,
    'decoder_mode': 1, #  swin transformer
    'init_audio_with_video_mae': False,
    'no_shift': False,
    ##

    'use_fbank': False,
    'fbank_dir': "/checkpoint/berniehuang/ast/egs/esc50/data/ESC-50-master/fbank",
    ##
    'data_path': '/datasets01/imagenet_full_size/061417/',
}

# 
data_train_relative = script_dir.joinpath(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_train.json'))
data_val_relative = script_dir.joinpath(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_val.json'))
label_csv_relative = script_dir.joinpath(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_labels_indices.csv'))
output_dir_relative = script_dir.joinpath(Path.cwd().parents[0].joinpath('output_pretrain_model', 'model_new'))

# 
args['data_train'] = str(data_train_relative)
args['data_val'] = str(data_val_relative)
args['label_csv'] = str(label_csv_relative)
args['output_dir'] = str(output_dir_relative)
args['log_dir'] = str(output_dir_relative)

os.environ['OMPI_COMM_WORLD_RANK'] = str(args['rank'])
os.environ['OMPI_COMM_WORLD_SIZE'] = str(args['world_size'])

In [4]:
# 
misc.init_distributed_mode(args)
# misc.init_distributed_mode(args['dist_on_itp'])

print('======================= starting pretrain =======================')
# print('job dir: {}'.format(os.path.dirname(os.path.realpath(__file__))))
cwd = os.getcwd()
print('Current directory:', cwd)
print("{}".format(args).replace(', ', ',\n'))

device = torch.device(args['device'])

# seed = args.seed + misc.get_rank()
seed = args['seed'] + misc.get_rank()
torch.manual_seed(seed)
np.random.seed(seed)

cudnn.benchmark = True

Not using distributed mode
[12:19:36.725257] ======================= starting pretrain =======================
[12:19:36.725469] Current directory: /media/mtshi/Windows-SSD/Linux/model/AudioMAE/code
[12:19:36.725551] {'batch_size': 1,
'epochs': 30,
'accum_iter': 1,
'model': 'mae_vit_base_patch16',
'mask_ratio': 0.8,
'norm_pix_loss': True,
'weight_decay': 0.0001,
'lr': None,
'blr': 0.0002,
'min_lr': 0.0,
'warmup_epochs': 4,
'input_size': 224,
'data_path': '/datasets01/imagenet_full_size/061417/',
'device': 'cuda',
'seed': 0,
'resume': '',
'start_epoch': 0,
'num_workers': 3,
'pin_mem': True,
'no_pin_mem': False,
'world_size': 1,
'local_rank': -1,
'dist_on_itp': False,
'dist_url': 'env://',
'audio_exp': True,
'freqm': 0,
'timem': 0,
'mixup': 0,
'dataset': 'birdsong',
'use_fbank': False,
'fbank_dir': '/checkpoint/berniehuang/ast/egs/esc50/data/ESC-50-master/fbank',
'alpha': 0.0,
'omega': 1.0,
'mode': 0,
'save_every_epoch': 1,
'use_custom_patch': False,
'roll_mag_aug': True,
'split_pos': Fa

In [5]:
# Select dataset
if not args['audio_exp']:
    dataset_train = BirdsongDataset(Path.cwd().parent.parent.joinpath('data', 'tmp', 'ae-train.csv'),
                                    needAugment=False,
                                    needLabel=False
                                    )
    
    dataset_val = BirdsongDataset(Path.cwd().parent.parent.joinpath('data', 'tmp', 'ae-validate.csv'),
                                  needAugment=False,
                                  needLabel=False
                                  )

else:
    norm_stats = {'birdsong':[args['mean'], args['std']]}
    target_length = {'birdsong':args['input_size']}
    multilabel_dataset = {'birdsong': True}
    audio_conf = {'num_mel_bins': 128, 
                  'target_length': target_length[args['dataset']], 
                  'freqm': args['freqm'],
                  'timem': args['timem'],
                  'mixup': args['mixup'],
                  'dataset': args['dataset'],
                  'mode':'train',
                  'mean': args['mean'],
                  'std': args['std'],
                  'multilabel': multilabel_dataset[args['dataset']],
                  'noise': args['noise'],
                  'frame_shift_ms': 10, # 7.687
                  'low_freq': args['low_freq'],
                  'high_freq': args['high_freq']
                  }
    
    val_audio_conf = {'num_mel_bins': 128,
                      'target_length': target_length[args['dataset']], 
                      'freqm': 0,
                      'timem': 0,
                      'mixup': 0,
                      'dataset': args['dataset'],
                      'mode':'val',
                      'mean': args['mean'],
                      'std': args['std'],
                      'multilabel': multilabel_dataset[args['dataset']],
                      'noise': False,
                      'frame_shift_ms': 10, # 7.687
                      'low_freq': args['low_freq'],
                      'high_freq': args['high_freq']
                      }

    dataset_train = AudiosetDataset(
                        args['data_train'],
                        label_csv = args['label_csv'],
                        audio_conf = audio_conf,
                        roll_mag_aug = args['roll_mag_aug'],
                        load_video = args['load_video']
                    )
    
    dataset_val = AudiosetDataset(
                        args['data_val'],
                        label_csv = args['label_csv'],
                        audio_conf = val_audio_conf,
                        roll_mag_aug = args['roll_mag_aug'],
                        load_video = args['load_video']
                    )


[12:19:43.711496] ---------------the train dataloader---------------
[12:19:43.711566] multilabel: True
[12:19:43.711573] using following mask: 0 freq, 0 time
[12:19:43.711580] using mix-up with rate 0.000000
[12:19:43.711589] use dataset mean -13.851 and std 3.861 to normalize the input.
[12:19:43.721010] number of classes: 32
[12:19:43.721022] size of dataset 3690474
[12:19:44.856573] ---------------the val dataloader---------------
[12:19:44.856644] multilabel: True
[12:19:44.856651] using following mask: 0 freq, 0 time
[12:19:44.856658] using mix-up with rate 0.000000
[12:19:44.856667] use dataset mean -13.851 and std 3.861 to normalize the input.
[12:19:44.856920] number of classes: 32
[12:19:44.856927] size of dataset 922618


In [6]:
if True:  # args.distributed:
    num_tasks = misc.get_world_size()
    global_rank = misc.get_rank()
    sampler_train = torch.utils.data.DistributedSampler(
                            dataset_train,
                            num_replicas = num_tasks,
                            rank = global_rank,
                            shuffle = True
                        )

    sampler_val = torch.utils.data.DistributedSampler(
                        dataset_val,
                        num_replicas = num_tasks,
                        rank = global_rank,
                        shuffle = False
                    )
    
    print("Sampler_train = %s" % str(sampler_train))

else:
    sampler_train = torch.utils.data.RandomSampler(dataset_train)
    sampler_val = torch.utils.data.SequentialSampler(dataset_val)

[12:19:48.254212] Sampler_train = <torch.utils.data.distributed.DistributedSampler object at 0x70330139bac0>


In [7]:
if global_rank == 0 and args['log_dir'] is not None:
    os.makedirs(args['log_dir'], exist_ok = True)
    log_writer = SummaryWriter(log_dir = args['log_dir'])
else:
    log_writer = None

In [8]:
# dataloader
data_loader_train = torch.utils.data.DataLoader(
                        dataset_train,
                        sampler = sampler_train,
                        batch_size = args['batch_size'],
                        num_workers = args['num_workers'],
                        pin_memory = args['pin_mem'],
                        drop_last = True,
                    )

data_loader_val = torch.utils.data.DataLoader(
                        dataset_val,
                        sampler = sampler_val,
                        batch_size = args['batch_size'],
                        num_workers = args['num_workers'],
                        pin_memory = args['pin_mem'],
                        drop_last = False,
                    )

In [9]:
# import masked auto encoder 
if args['audio_exp']:
    model = models_mae.__dict__[args['model']](
        norm_pix_loss = args['norm_pix_loss'],
        in_chans = 1,
        audio_exp = True,
        img_size = (target_length[args['dataset']], 128),
        alpha = args['alpha'],
        mode = args['mode'],
        use_custom_patch = args['use_custom_patch'],
        split_pos = args['split_pos'],
        pos_trainable = args['pos_trainable'],
        use_nce = args['use_nce'],
        decoder_mode = args['decoder_mode'],
        mask_2d = args['mask_2d'],
        mask_t_prob = args['mask_t_prob'],
        mask_f_prob = args['mask_f_prob'],
        no_shift = args['no_shift'],
        # remove for A-MAE
        # v_weight=args.v_weight, n_frm=args.n_frm, video_only=args.video_only, cl=args.cl, depth_av=args.depth_av,
    )
    
else:
    model = models_mae.__dict__[args['model']](norm_pix_loss = args['norm_pix_loss'])

model.to(device)
model_without_ddp = model
print("Model = %s" % str(model_without_ddp))

[12:20:09.539424] Model = MaskedAutoencoderViT(
  (patch_embed): PatchEmbed_org(
    (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU()
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linea

In [10]:
eff_batch_size = args['batch_size'] * args['accum_iter'] * misc.get_world_size()
    
if args['lr'] is None:  # only base_lr is specified
    args['lr'] = args['blr'] * eff_batch_size / 256

print("base lr: %.2e" % (args['lr'] * 256 / eff_batch_size))
print("actual lr: %.2e" % args['lr'])

print("accumulate grad iterations: %d" % args['accum_iter'])
print("effective batch size: %d" % eff_batch_size)

[12:20:11.721703] base lr: 2.00e-04
[12:20:11.721819] actual lr: 7.81e-07
[12:20:11.721868] accumulate grad iterations: 1
[12:20:11.721912] effective batch size: 1


In [11]:
# 
if args['distributed']:
    print('use distributed!!')
    model = torch.nn.parallel.DistributedDataParallel(
                model,
                device_ids=[args['gpu']],
                find_unused_parameters = True
            )
    
    model_without_ddp = model.module

In [12]:

# following timm: set wd as 0 for bias and norm layers
param_groups = optim_factory.add_weight_decay(
                    model_without_ddp,
                    args['weight_decay']
                )

optimizer = torch.optim.AdamW(
                param_groups,
                lr = args['lr'],
                betas = (0.9, 0.95)
            )

print(optimizer)

loss_scaler = NativeScaler()

misc.load_model(
    args = args,
    model_without_ddp = model_without_ddp,
    optimizer = optimizer,
    loss_scaler = loss_scaler
)

print(f"Start training for {args['epochs']} epochs")
start_time = time.time()

[12:20:34.365762] AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    eps: 1e-08
    lr: 7.8125e-07
    weight_decay: 0.0

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    eps: 1e-08
    lr: 7.8125e-07
    weight_decay: 0.0001
)
[12:20:34.365922] Start training for 30 epochs


In [ ]:
for epoch in range(args['start_epoch'], args['epochs']):

    if args['distributed']:
        data_loader_train.sampler.set_epoch(epoch)

    # training
    train_stats = train_one_epoch(
                        model,
                        data_loader_train,
                        optimizer,
                        device,
                        epoch,
                        loss_scaler,
                        log_writer = log_writer,
                        args = args
                    )

    # validation
    val_stats = validate_one_epoch(
                    model,
                    data_loader_val,
                    device,
                    epoch,
                    log_writer = log_writer,
                    args = args
                )

    if args['output_dir'] and (epoch % args['save_every_epoch'] == 0 or epoch + 1 == args['epochs']):
        misc.save_model(
            args = args,
            model = model,
            model_without_ddp = model_without_ddp,
            optimizer = optimizer,
            loss_scaler = loss_scaler,
            epoch = epoch
        )

    log_stats = {**{f'train_{k}': v for k, v in train_stats.items()},
                 **{f'val_{k}': v for k, v in val_stats.items()},
                 'epoch': epoch,
                 }

    if args['output_dir'] and misc.is_main_process():
        if log_writer is not None:
            log_writer.flush()
        with open(os.path.join(args['output_dir'], "log.txt"), mode = "a", encoding = "utf-8") as f:
            f.write(json.dumps(log_stats) + "\n")

total_time = time.time() - start_time
total_time_str = str(datetime.timedelta(seconds = int(total_time)))
print('Training time {}'.format(total_time_str))